# Run EvoProtGrad on nucB dataset

In [ ]:
# Load packages and define ref_seq for Mut321
%run setup_environment.py
import seq2fitness_models as models
import evo_prot_grad
import seq2fitness_expert
from seq2fitness_expert import build_custom_expert
import evo_prot_grad.common.sampler as sampler
import time
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
# Convert ref_seq to Fasta format
def convert_to_fasta(sequence, description="Sequence"):
    """Convert an amino acid sequence string to FASTA format."""
    fasta_format = f">{description}\n"
    for i in range(0, len(sequence), 60):
        #fasta_format += sequence[i:i+60] + "\n")
        fasta_format += sequence[i:i+60]
    return fasta_format

# Convert ref_seq to FASTA format
fasta_content = convert_to_fasta(ref_seq_nucb, description="nucB")

# Save FASTA content to a file
with open("ref_seq_nucB.fasta", "w") as file:
    file.write(fasta_content)

In [ ]:

# Define the model checkpoint path
model_path = '../../../../trained_models/nucb_model_final_model_epoch_100_val_loss_0.5445.pt'

# Define the temperature and device
temperature = 1.0
device = 'cuda:0' if torch.cuda.is_available() else 'cpu'

new_task_weights = {
    'activity_greater_than_neg_control': 0.000,
    'activity_greater_than_wt': 1.000,
    'activity_greater_than_a73r': 0.000
}

# Build the custom expert
expert = build_custom_expert(model_path, temperature, device, new_task_weights)

# Initialize the wildtype sequence
expert.init_wildtype(ref_seq_nucb)
wt_score = expert._wt_score

In [ ]:
fasta_path = "ref_seq_nucB.fasta"
# Initialize DirectedEvolution
n_steps = 3000 # max seems to be 4500 before we run out of memory
parallel_chains = 8 # max is 4 before our GPU runs out of RAM. For 4500 steps with 4 chains, takes half an hour.
max_mutations = 5
evolution = evo_prot_grad.DirectedEvolution(
    experts=[expert],
    parallel_chains=parallel_chains, # Max the RTX 4090 can handle is 5. Out of memory if more.
    n_steps=n_steps,
    max_mutations=max_mutations,
    output='all',
    wt_fasta=fasta_path,
    verbose=False,
    preserved_regions = [(1, 1)], # sites to exclude. ESM is bad at site 1, and 330 mutated already for the better
    random_seed=7
)
start_time = time.time()
# Run evolution
variants, scores = evolution()
end_time = time.time()
elapsed_time = end_time - start_time
print(f"Elapsed time: {elapsed_time:.2f} seconds")

In [ ]:
# Save the results to a CSV file
fname = f'original_evo_prot_grad_results_max_mutations_{max_mutations}_nsteps_{n_steps}_parallel_chains_{parallel_chains}.csv'
evolution.save_results(fname, variants, scores)